# 28. Maximize Items Under Budget

**URL:** https://www.sparkplayground.com/pyspark-coding-interview-questions/maximize-items-under-budget  
**Difficulty:** Medium  
**Date:** 2026-04-26

## Task
Given products with `price` and available `quantity`, and a budget of **$100**, return the **maximum number of items** that can be purchased.
- Buy at most `quantity` units of each product.
- Prices are per unit; units are indivisible (whole items only).
- Maximize total item count subject to total spend ≤ 100.

## Approach — cheap-first greedy
Every item contributes the same value (= 1) to the count, so the lowest-cost item always gives the best value-per-dollar. Sort by `price` ascending, take the full quantity while the running total stays ≤ budget, and on the **boundary row** (where it first crosses 100) take only as many units as the remaining budget allows.

Implemented as a single window-function pass instead of a Python loop — keeps the work in Spark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder.getOrCreate()

data = [
    ("Eraser", 1, 9),
    ("Pen", 10, 5),
    ("Book", 40, 3),
    ("Bag", 60, 2),
    ("Pencil", 5, 10),
    ("Bottle", 30, 4),
    ("Headphones", 80, 1),
    ("Notebook", 15, 6),
]
columns = ["product", "price", "quantity"]
df = spark.createDataFrame(data, columns)
df.show()

## Solution — window-function greedy

1. `amount = price * quantity` — cost if we bought the entire stock of this row.
2. `running_total` = cumulative cost in cheap-first order.
3. `previous_total` = running total **excluding** the current row — i.e. budget already spent before reaching this product.
4. Per-row decision:
   - `running_total ≤ 100` → take the full `quantity`.
   - else if `previous_total < 100` → boundary row, take `floor((100 - previous_total) / price)` units.
   - else → budget already spent, take 0.

Tiebreak by `product` so the order is deterministic when two items share a price.

In [ ]:
window_spec = (
    Window
    .orderBy("price", "product")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df = (
    df
    .withColumn("amount", F.col("price") * F.col("quantity"))
    .withColumn("running_total", F.sum("amount").over(window_spec))
    .withColumn("previous_total", F.col("running_total") - F.col("amount"))
    .withColumn(
        "item_count",
        F.when(F.col("running_total") <= 100, F.col("quantity"))
         .when(F.col("previous_total") < 100,
               F.floor((100 - F.col("previous_total")) / F.col("price")))
         .otherwise(0),
    )
)

df.orderBy("price", "product").show()

df_result = df.agg(
    F.sum("item_count").alias("max_items"),
    F.sum(F.col("item_count") * F.col("price")).cast("int").alias("total_spent"),
)
df_result.show()

## Notes

- **Why greedy is optimal here:** every item contributes the same value (1) to the objective, so the swap argument holds — replacing any expensive unit with a cheaper one never decreases the count and never exceeds the budget. This is the classic "unit-value knapsack" → sort by cost ascending.
- **Boundary row math:** `previous_total` is the spend right before this product. Remaining budget is `100 - previous_total`, and each unit of this product costs `price`, so `floor((100 - previous_total) / price)` is the most units that still fit. `floor` matters because units are indivisible.
- **Why a window, not a Python loop:** the running total + per-row decision is exactly what `Window.rowsBetween(unboundedPreceding, currentRow)` was built for. Stays in the JVM, scales to large product catalogs.
- **What this is NOT:** not a true bounded-knapsack DP. Greedy works *only* because all values are equal. If items had different `value` fields, the greedy by price would no longer be optimal and you'd need DP (or for very small budgets, an exhaustive search).